# Task4 — Backbone diversity after advanced EDA

เป้าหมาย: ลด shared errors ของ ResNet18 ด้วย EfficientNet-B0 โดยใช้ compound augmentation เดิม, weighted CE, 18 epochs คงที่ และ 5 folds เดิม

Run All: ตรวจ Train → rectangular inference probe → ฝึก EfficientNet 5 folds → paired OOF/stress → audit → full refit เฉพาะเมื่อผ่านเกณฑ์ ไม่เปิด Test และไม่สร้าง submission อัตโนมัติ

มีเพียง **หนึ่ง backbone ใหม่: 5 fold models + 1 full model ถ้าผ่าน** ไม่มี epoch selection จาก validation. EDA/OOF ถูกใช้หลายครั้งแล้ว ผลเปรียบเทียบเป็น exploratory ไม่ใช่ independent validation หรือการรับรอง Public score.

EfficientNet ใช้ ImageNet weights ที่กติกาอนุญาตสำหรับ Task4; รอบแรกอาจดาวน์โหลด weights ผ่าน torchvision. การฝึกใช้เฉพาะ Train ของการแข่งขัน.

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_diversity_training as training
from Task4.models import image_diversity_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/backbone_diversity_v1'
BATCH_SIZE=32
NUM_WORKERS=0
RUN_RECTANGULAR_PROBE=True
RUN_FULL_REFIT=True
NAME='efficientnet_compound18'
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')
display(asdict(training.RECIPES[NAME]))

Device: mps


{'architecture': 'efficientnet_b0',
 'epochs': 18,
 'backbone_lr': 0.0001,
 'head_lr': 0.0005,
 'weight_decay': 0.0001,
 'schedule': 'cosine',
 'warmup_epochs': 1,
 'augmentation': 'compound_corruption',
 'label_smoothing': 0.0,
 'seed': 42,
 'balance': 'weighted_loss',
 'jsd_weight': 0.0,
 'compound_probability': 0.6}

## 1. Verify Train, fold identity และแหล่งข้อมูล
ตรวจไฟล์ Train กับ audit hashes เดิม ไม่แตะ Test. ถ้า settings เปลี่ยนให้ใช้ RUN directory ใหม่ ห้าม resume ปะปน config.

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':asdict(training.RECIPES[NAME]),'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. Rectangular inference probe — ยังไม่ใช่ rectangular retraining
ใช้ fold1 models เดิม 2 ตัว เทียบ identity view แบบ padding กับ 112×224 ไม่มี padding. ไม่มีการเปลี่ยน contrast และไม่เลือกค่าใดจาก Test. ผลไม่ถูกใช้ปรับ EfficientNet recipe: โมเดลเหล่านี้ฝึกด้วย padding จึงมี preprocessing shift เมื่อเอา padding ออก.

In [3]:
if RUN_RECTANGULAR_PROBE:
    display(evaluation.rectangular_probe(ROOT,RAW,groups,folds,OUT,BATCH_SIZE))

,preprocessing,macro_F1,F1_7-11k,F1_fiveN,F1_freefried,F1_pube,F1_roblock
0,padded_identity,0.981144,0.989956,0.978385,0.990629,0.984368,0.962382
1,rectangle_identity,0.965275,0.983474,0.965208,0.973974,0.954565,0.949153


## 3. Train fixed 18 epochs × 5 folds และ paired validation
Checkpoint `resume.pt` ทุก epoch; `model.pt` หลังจบ. Interrupt แล้วรันใหม่จะกลับจาก epoch ล่าสุดที่เสร็จ (epoch ค้างต้องทำใหม่). GPU utilization/ความร้อนขึ้นได้ตาม workload.

วัด baseline compound+sampler, EfficientNet เดี่ยว, และ blend50 ที่กำหนดล่วงหน้า (50% baseline + 50% EfficientNet). ไม่ค้นน้ำหนัก ensemble หลายค่า. Stress เป็นชุด Train-derived เดิม; ไม่เท่ากับ distribution ของ Test. เวลาจริงให้ดู seconds ของ epoch แรก; จำนวน epoch ฝึกรวม 90 และอีก 18 หาก full refit.

In [4]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/meng/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.23MB/s]


fold1 epoch 1 / 18 loss 0.75933 seconds 214.4
fold1 epoch 2 / 18 loss 0.40118 seconds 206.2
fold1 epoch 3 / 18 loss 0.30187 seconds 205.6
fold1 epoch 4 / 18 loss 0.23698 seconds 205.8
fold1 epoch 5 / 18 loss 0.20993 seconds 206.1
fold1 epoch 6 / 18 loss 0.18377 seconds 204.7
fold1 epoch 7 / 18 loss 0.15754 seconds 204.9
fold1 epoch 8 / 18 loss 0.14063 seconds 204.9
fold1 epoch 9 / 18 loss 0.13002 seconds 205.1
fold1 epoch 10 / 18 loss 0.11226 seconds 205.2
fold1 epoch 11 / 18 loss 0.10062 seconds 205.2
fold1 epoch 12 / 18 loss 0.09453 seconds 206.1
fold1 epoch 13 / 18 loss 0.08585 seconds 206.1
fold1 epoch 14 / 18 loss 0.0831 seconds 207.0
fold1 epoch 15 / 18 loss 0.07908 seconds 205.6
fold1 epoch 16 / 18 loss 0.07341 seconds 207.5
fold1 epoch 17 / 18 loss 0.07428 seconds 208.8
fold1 epoch 18 / 18 loss 0.07123 seconds 208.0
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fol

suite,compound,original,single
procedure,,,
baseline,0.929587,0.981367,0.975196
blend50,0.944725,0.986331,0.977201
efficientnet,0.937731,0.981883,0.973776


Approx remaining CV TRAIN minutes (excludes validation): 249.7

EfficientNet fold 2/5
fold2 epoch 1 / 18 loss 0.75993 seconds 209.0
fold2 epoch 2 / 18 loss 0.3767 seconds 206.4
fold2 epoch 3 / 18 loss 0.29719 seconds 203.8
fold2 epoch 4 / 18 loss 0.23959 seconds 205.8
fold2 epoch 5 / 18 loss 0.20431 seconds 204.9
fold2 epoch 6 / 18 loss 0.17661 seconds 205.1
fold2 epoch 7 / 18 loss 0.16118 seconds 206.6
fold2 epoch 8 / 18 loss 0.14294 seconds 205.2
fold2 epoch 9 / 18 loss 0.12619 seconds 205.4
fold2 epoch 10 / 18 loss 0.11078 seconds 205.2
fold2 epoch 11 / 18 loss 0.10463 seconds 205.1
fold2 epoch 12 / 18 loss 0.09125 seconds 205.0
fold2 epoch 13 / 18 loss 0.08875 seconds 205.1
fold2 epoch 14 / 18 loss 0.08198 seconds 205.0
fold2 epoch 15 / 18 loss 0.07566 seconds 204.8
fold2 epoch 16 / 18 loss 0.07024 seconds 205.1
fold2 epoch 17 / 18 loss 0.07185 seconds 205.1
fold2 epoch 18 / 18 loss 0.07405 seconds 205.4
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress 

suite,compound,original,single
procedure,,,
baseline,0.936276,0.980392,0.982209
blend50,0.955820,0.983375,0.985600
efficientnet,0.956336,0.980697,0.984579


... (3 outputs left out)

suite,compound,original,single
procedure,,,
baseline,0.925264,0.978785,0.972586
blend50,0.942175,0.980890,0.981004
efficientnet,0.942818,0.979768,0.981404


Approx remaining CV TRAIN minutes (excludes validation): 62.4

EfficientNet fold 5/5
fold5 epoch 1 / 18 loss 0.73865 seconds 207.8
fold5 epoch 2 / 18 loss 0.39001 seconds 207.8
fold5 epoch 3 / 18 loss 0.28509 seconds 212.7
fold5 epoch 4 / 18 loss 0.23898 seconds 209.9
fold5 epoch 5 / 18 loss 0.20023 seconds 207.5
fold5 epoch 6 / 18 loss 0.17203 seconds 205.4
fold5 epoch 7 / 18 loss 0.15233 seconds 204.7
fold5 epoch 8 / 18 loss 0.138 seconds 204.6
fold5 epoch 9 / 18 loss 0.12059 seconds 204.9
fold5 epoch 10 / 18 loss 0.11771 seconds 204.9
fold5 epoch 11 / 18 loss 0.10082 seconds 204.6
fold5 epoch 12 / 18 loss 0.094 seconds 209.4
fold5 epoch 13 / 18 loss 0.09015 seconds 217.8
fold5 epoch 14 / 18 loss 0.08068 seconds 210.6
fold5 epoch 15 / 18 loss 0.07268 seconds 212.8
fold5 epoch 16 / 18 loss 0.06977 seconds 213.4
fold5 epoch 17 / 18 loss 0.06896 seconds 212.5
fold5 epoch 18 / 18 loss 0.06841 seconds 213.1
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1
stress fold

suite,compound,original,single
procedure,,,
baseline,0.919297,0.980762,0.973996
blend50,0.940842,0.984526,0.981418
efficientnet,0.936064,0.981459,0.977605


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.44


In [ ]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/meng/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.23MB/s]


fold1 epoch 1 / 18 loss 0.75933 seconds 214.4
fold1 epoch 2 / 18 loss 0.40118 seconds 206.2
fold1 epoch 3 / 18 loss 0.30187 seconds 205.6
fold1 epoch 4 / 18 loss 0.23698 seconds 205.8
fold1 epoch 5 / 18 loss 0.20993 seconds 206.1
fold1 epoch 6 / 18 loss 0.18377 seconds 204.7
fold1 epoch 7 / 18 loss 0.15754 seconds 204.9
fold1 epoch 8 / 18 loss 0.14063 seconds 204.9
fold1 epoch 9 / 18 loss 0.13002 seconds 205.1
fold1 epoch 10 / 18 loss 0.11226 seconds 205.2
fold1 epoch 11 / 18 loss 0.10062 seconds 205.2
fold1 epoch 12 / 18 loss 0.09453 seconds 206.1
fold1 epoch 13 / 18 loss 0.08585 seconds 206.1
fold1 epoch 14 / 18 loss 0.0831 seconds 207.0
fold1 epoch 15 / 18 loss 0.07908 seconds 205.6
fold1 epoch 16 / 18 loss 0.07341 seconds 207.5
fold1 epoch 17 / 18 loss 0.07428 seconds 208.8
fold1 epoch 18 / 18 loss 0.07123 seconds 208.0
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fol

suite,compound,original,single
procedure,,,
baseline,0.929587,0.981367,0.975196
blend50,0.944725,0.986331,0.977201
efficientnet,0.937731,0.981883,0.973776


Approx remaining CV TRAIN minutes (excludes validation): 249.7

EfficientNet fold 2/5
fold2 epoch 1 / 18 loss 0.75993 seconds 209.0
fold2 epoch 2 / 18 loss 0.3767 seconds 206.4
fold2 epoch 3 / 18 loss 0.29719 seconds 203.8
fold2 epoch 4 / 18 loss 0.23959 seconds 205.8
fold2 epoch 5 / 18 loss 0.20431 seconds 204.9
fold2 epoch 6 / 18 loss 0.17661 seconds 205.1
fold2 epoch 7 / 18 loss 0.16118 seconds 206.6
fold2 epoch 8 / 18 loss 0.14294 seconds 205.2
fold2 epoch 9 / 18 loss 0.12619 seconds 205.4
fold2 epoch 10 / 18 loss 0.11078 seconds 205.2
fold2 epoch 11 / 18 loss 0.10463 seconds 205.1
fold2 epoch 12 / 18 loss 0.09125 seconds 205.0
fold2 epoch 13 / 18 loss 0.08875 seconds 205.1
fold2 epoch 14 / 18 loss 0.08198 seconds 205.0
fold2 epoch 15 / 18 loss 0.07566 seconds 204.8
fold2 epoch 16 / 18 loss 0.07024 seconds 205.1
fold2 epoch 17 / 18 loss 0.07185 seconds 205.1
fold2 epoch 18 / 18 loss 0.07405 seconds 205.4
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress 

suite,compound,original,single
procedure,,,
baseline,0.936276,0.980392,0.982209
blend50,0.955820,0.983375,0.985600
efficientnet,0.956336,0.980697,0.984579


... (3 outputs left out)

suite,compound,original,single
procedure,,,
baseline,0.925264,0.978785,0.972586
blend50,0.942175,0.980890,0.981004
efficientnet,0.942818,0.979768,0.981404


Approx remaining CV TRAIN minutes (excludes validation): 62.4

EfficientNet fold 5/5
fold5 epoch 1 / 18 loss 0.73865 seconds 207.8
fold5 epoch 2 / 18 loss 0.39001 seconds 207.8
fold5 epoch 3 / 18 loss 0.28509 seconds 212.7
fold5 epoch 4 / 18 loss 0.23898 seconds 209.9
fold5 epoch 5 / 18 loss 0.20023 seconds 207.5
fold5 epoch 6 / 18 loss 0.17203 seconds 205.4
fold5 epoch 7 / 18 loss 0.15233 seconds 204.7
fold5 epoch 8 / 18 loss 0.138 seconds 204.6
fold5 epoch 9 / 18 loss 0.12059 seconds 204.9
fold5 epoch 10 / 18 loss 0.11771 seconds 204.9
fold5 epoch 11 / 18 loss 0.10082 seconds 204.6
fold5 epoch 12 / 18 loss 0.094 seconds 209.4
fold5 epoch 13 / 18 loss 0.09015 seconds 217.8
fold5 epoch 14 / 18 loss 0.08068 seconds 210.6
fold5 epoch 15 / 18 loss 0.07268 seconds 212.8
fold5 epoch 16 / 18 loss 0.06977 seconds 213.4
fold5 epoch 17 / 18 loss 0.06896 seconds 212.5
fold5 epoch 18 / 18 loss 0.06841 seconds 213.1
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1
stress fold

suite,compound,original,single
procedure,,,
baseline,0.919297,0.980762,0.973996
blend50,0.940842,0.984526,0.981418
efficientnet,0.936064,0.981459,0.977605


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.44


In [ ]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/meng/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.23MB/s]


fold1 epoch 1 / 18 loss 0.75933 seconds 214.4
fold1 epoch 2 / 18 loss 0.40118 seconds 206.2
fold1 epoch 3 / 18 loss 0.30187 seconds 205.6
fold1 epoch 4 / 18 loss 0.23698 seconds 205.8
fold1 epoch 5 / 18 loss 0.20993 seconds 206.1
fold1 epoch 6 / 18 loss 0.18377 seconds 204.7
fold1 epoch 7 / 18 loss 0.15754 seconds 204.9
fold1 epoch 8 / 18 loss 0.14063 seconds 204.9
fold1 epoch 9 / 18 loss 0.13002 seconds 205.1
fold1 epoch 10 / 18 loss 0.11226 seconds 205.2
fold1 epoch 11 / 18 loss 0.10062 seconds 205.2
fold1 epoch 12 / 18 loss 0.09453 seconds 206.1
fold1 epoch 13 / 18 loss 0.08585 seconds 206.1
fold1 epoch 14 / 18 loss 0.0831 seconds 207.0
fold1 epoch 15 / 18 loss 0.07908 seconds 205.6
fold1 epoch 16 / 18 loss 0.07341 seconds 207.5
fold1 epoch 17 / 18 loss 0.07428 seconds 208.8
fold1 epoch 18 / 18 loss 0.07123 seconds 208.0
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fol

suite,compound,original,single
procedure,,,
baseline,0.929587,0.981367,0.975196
blend50,0.944725,0.986331,0.977201
efficientnet,0.937731,0.981883,0.973776


Approx remaining CV TRAIN minutes (excludes validation): 249.7

EfficientNet fold 2/5
fold2 epoch 1 / 18 loss 0.75993 seconds 209.0
fold2 epoch 2 / 18 loss 0.3767 seconds 206.4
fold2 epoch 3 / 18 loss 0.29719 seconds 203.8
fold2 epoch 4 / 18 loss 0.23959 seconds 205.8
fold2 epoch 5 / 18 loss 0.20431 seconds 204.9
fold2 epoch 6 / 18 loss 0.17661 seconds 205.1
fold2 epoch 7 / 18 loss 0.16118 seconds 206.6
fold2 epoch 8 / 18 loss 0.14294 seconds 205.2
fold2 epoch 9 / 18 loss 0.12619 seconds 205.4
fold2 epoch 10 / 18 loss 0.11078 seconds 205.2
fold2 epoch 11 / 18 loss 0.10463 seconds 205.1
fold2 epoch 12 / 18 loss 0.09125 seconds 205.0
fold2 epoch 13 / 18 loss 0.08875 seconds 205.1
fold2 epoch 14 / 18 loss 0.08198 seconds 205.0
fold2 epoch 15 / 18 loss 0.07566 seconds 204.8
fold2 epoch 16 / 18 loss 0.07024 seconds 205.1
fold2 epoch 17 / 18 loss 0.07185 seconds 205.1
fold2 epoch 18 / 18 loss 0.07405 seconds 205.4
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress 

suite,compound,original,single
procedure,,,
baseline,0.936276,0.980392,0.982209
blend50,0.955820,0.983375,0.985600
efficientnet,0.956336,0.980697,0.984579


... (3 outputs left out)

suite,compound,original,single
procedure,,,
baseline,0.925264,0.978785,0.972586
blend50,0.942175,0.980890,0.981004
efficientnet,0.942818,0.979768,0.981404


Approx remaining CV TRAIN minutes (excludes validation): 62.4

EfficientNet fold 5/5
fold5 epoch 1 / 18 loss 0.73865 seconds 207.8
fold5 epoch 2 / 18 loss 0.39001 seconds 207.8
fold5 epoch 3 / 18 loss 0.28509 seconds 212.7
fold5 epoch 4 / 18 loss 0.23898 seconds 209.9
fold5 epoch 5 / 18 loss 0.20023 seconds 207.5
fold5 epoch 6 / 18 loss 0.17203 seconds 205.4
fold5 epoch 7 / 18 loss 0.15233 seconds 204.7
fold5 epoch 8 / 18 loss 0.138 seconds 204.6
fold5 epoch 9 / 18 loss 0.12059 seconds 204.9
fold5 epoch 10 / 18 loss 0.11771 seconds 204.9
fold5 epoch 11 / 18 loss 0.10082 seconds 204.6
fold5 epoch 12 / 18 loss 0.094 seconds 209.4
fold5 epoch 13 / 18 loss 0.09015 seconds 217.8
fold5 epoch 14 / 18 loss 0.08068 seconds 210.6
fold5 epoch 15 / 18 loss 0.07268 seconds 212.8
fold5 epoch 16 / 18 loss 0.06977 seconds 213.4
fold5 epoch 17 / 18 loss 0.06896 seconds 212.5
fold5 epoch 18 / 18 loss 0.06841 seconds 213.1
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1
stress fold

suite,compound,original,single
procedure,,,
baseline,0.919297,0.980762,0.973996
blend50,0.940842,0.984526,0.981418
efficientnet,0.936064,0.981459,0.977605


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.44


In [ ]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/meng/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.23MB/s]


fold1 epoch 1 / 18 loss 0.75933 seconds 214.4
fold1 epoch 2 / 18 loss 0.40118 seconds 206.2
fold1 epoch 3 / 18 loss 0.30187 seconds 205.6
fold1 epoch 4 / 18 loss 0.23698 seconds 205.8
fold1 epoch 5 / 18 loss 0.20993 seconds 206.1
fold1 epoch 6 / 18 loss 0.18377 seconds 204.7
fold1 epoch 7 / 18 loss 0.15754 seconds 204.9
fold1 epoch 8 / 18 loss 0.14063 seconds 204.9
fold1 epoch 9 / 18 loss 0.13002 seconds 205.1
fold1 epoch 10 / 18 loss 0.11226 seconds 205.2
fold1 epoch 11 / 18 loss 0.10062 seconds 205.2
fold1 epoch 12 / 18 loss 0.09453 seconds 206.1
fold1 epoch 13 / 18 loss 0.08585 seconds 206.1
fold1 epoch 14 / 18 loss 0.0831 seconds 207.0
fold1 epoch 15 / 18 loss 0.07908 seconds 205.6
fold1 epoch 16 / 18 loss 0.07341 seconds 207.5
fold1 epoch 17 / 18 loss 0.07428 seconds 208.8
fold1 epoch 18 / 18 loss 0.07123 seconds 208.0
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fol

suite,compound,original,single
procedure,,,
baseline,0.929587,0.981367,0.975196
blend50,0.944725,0.986331,0.977201
efficientnet,0.937731,0.981883,0.973776


Approx remaining CV TRAIN minutes (excludes validation): 249.7

EfficientNet fold 2/5
fold2 epoch 1 / 18 loss 0.75993 seconds 209.0
fold2 epoch 2 / 18 loss 0.3767 seconds 206.4
fold2 epoch 3 / 18 loss 0.29719 seconds 203.8
fold2 epoch 4 / 18 loss 0.23959 seconds 205.8
fold2 epoch 5 / 18 loss 0.20431 seconds 204.9
fold2 epoch 6 / 18 loss 0.17661 seconds 205.1
fold2 epoch 7 / 18 loss 0.16118 seconds 206.6
fold2 epoch 8 / 18 loss 0.14294 seconds 205.2
fold2 epoch 9 / 18 loss 0.12619 seconds 205.4
fold2 epoch 10 / 18 loss 0.11078 seconds 205.2
fold2 epoch 11 / 18 loss 0.10463 seconds 205.1
fold2 epoch 12 / 18 loss 0.09125 seconds 205.0
fold2 epoch 13 / 18 loss 0.08875 seconds 205.1
fold2 epoch 14 / 18 loss 0.08198 seconds 205.0
fold2 epoch 15 / 18 loss 0.07566 seconds 204.8
fold2 epoch 16 / 18 loss 0.07024 seconds 205.1
fold2 epoch 17 / 18 loss 0.07185 seconds 205.1
fold2 epoch 18 / 18 loss 0.07405 seconds 205.4
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress 

suite,compound,original,single
procedure,,,
baseline,0.936276,0.980392,0.982209
blend50,0.955820,0.983375,0.985600
efficientnet,0.956336,0.980697,0.984579


... (3 outputs left out)

suite,compound,original,single
procedure,,,
baseline,0.925264,0.978785,0.972586
blend50,0.942175,0.980890,0.981004
efficientnet,0.942818,0.979768,0.981404


Approx remaining CV TRAIN minutes (excludes validation): 62.4

EfficientNet fold 5/5
fold5 epoch 1 / 18 loss 0.73865 seconds 207.8
fold5 epoch 2 / 18 loss 0.39001 seconds 207.8
fold5 epoch 3 / 18 loss 0.28509 seconds 212.7
fold5 epoch 4 / 18 loss 0.23898 seconds 209.9
fold5 epoch 5 / 18 loss 0.20023 seconds 207.5
fold5 epoch 6 / 18 loss 0.17203 seconds 205.4
fold5 epoch 7 / 18 loss 0.15233 seconds 204.7
fold5 epoch 8 / 18 loss 0.138 seconds 204.6
fold5 epoch 9 / 18 loss 0.12059 seconds 204.9
fold5 epoch 10 / 18 loss 0.11771 seconds 204.9
fold5 epoch 11 / 18 loss 0.10082 seconds 204.6
fold5 epoch 12 / 18 loss 0.094 seconds 209.4
fold5 epoch 13 / 18 loss 0.09015 seconds 217.8
fold5 epoch 14 / 18 loss 0.08068 seconds 210.6
fold5 epoch 15 / 18 loss 0.07268 seconds 212.8
fold5 epoch 16 / 18 loss 0.06977 seconds 213.4
fold5 epoch 17 / 18 loss 0.06896 seconds 212.5
fold5 epoch 18 / 18 loss 0.06841 seconds 213.1
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1
stress fold

suite,compound,original,single
procedure,,,
baseline,0.919297,0.980762,0.973996
blend50,0.940842,0.984526,0.981418
efficientnet,0.936064,0.981459,0.977605


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.44


In [ ]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/meng/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.23MB/s]


fold1 epoch 1 / 18 loss 0.75933 seconds 214.4
fold1 epoch 2 / 18 loss 0.40118 seconds 206.2
fold1 epoch 3 / 18 loss 0.30187 seconds 205.6
fold1 epoch 4 / 18 loss 0.23698 seconds 205.8
fold1 epoch 5 / 18 loss 0.20993 seconds 206.1
fold1 epoch 6 / 18 loss 0.18377 seconds 204.7
fold1 epoch 7 / 18 loss 0.15754 seconds 204.9
fold1 epoch 8 / 18 loss 0.14063 seconds 204.9
fold1 epoch 9 / 18 loss 0.13002 seconds 205.1
fold1 epoch 10 / 18 loss 0.11226 seconds 205.2
fold1 epoch 11 / 18 loss 0.10062 seconds 205.2
fold1 epoch 12 / 18 loss 0.09453 seconds 206.1
fold1 epoch 13 / 18 loss 0.08585 seconds 206.1
fold1 epoch 14 / 18 loss 0.0831 seconds 207.0
fold1 epoch 15 / 18 loss 0.07908 seconds 205.6
fold1 epoch 16 / 18 loss 0.07341 seconds 207.5
fold1 epoch 17 / 18 loss 0.07428 seconds 208.8
fold1 epoch 18 / 18 loss 0.07123 seconds 208.0
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fol

suite,compound,original,single
procedure,,,
baseline,0.929587,0.981367,0.975196
blend50,0.944725,0.986331,0.977201
efficientnet,0.937731,0.981883,0.973776


Approx remaining CV TRAIN minutes (excludes validation): 249.7

EfficientNet fold 2/5
fold2 epoch 1 / 18 loss 0.75993 seconds 209.0
fold2 epoch 2 / 18 loss 0.3767 seconds 206.4
fold2 epoch 3 / 18 loss 0.29719 seconds 203.8
fold2 epoch 4 / 18 loss 0.23959 seconds 205.8
fold2 epoch 5 / 18 loss 0.20431 seconds 204.9
fold2 epoch 6 / 18 loss 0.17661 seconds 205.1
fold2 epoch 7 / 18 loss 0.16118 seconds 206.6
fold2 epoch 8 / 18 loss 0.14294 seconds 205.2
fold2 epoch 9 / 18 loss 0.12619 seconds 205.4
fold2 epoch 10 / 18 loss 0.11078 seconds 205.2
fold2 epoch 11 / 18 loss 0.10463 seconds 205.1
fold2 epoch 12 / 18 loss 0.09125 seconds 205.0
fold2 epoch 13 / 18 loss 0.08875 seconds 205.1
fold2 epoch 14 / 18 loss 0.08198 seconds 205.0
fold2 epoch 15 / 18 loss 0.07566 seconds 204.8
fold2 epoch 16 / 18 loss 0.07024 seconds 205.1
fold2 epoch 17 / 18 loss 0.07185 seconds 205.1
fold2 epoch 18 / 18 loss 0.07405 seconds 205.4
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress 

suite,compound,original,single
procedure,,,
baseline,0.936276,0.980392,0.982209
blend50,0.955820,0.983375,0.985600
efficientnet,0.956336,0.980697,0.984579


... (3 outputs left out)

suite,compound,original,single
procedure,,,
baseline,0.925264,0.978785,0.972586
blend50,0.942175,0.980890,0.981004
efficientnet,0.942818,0.979768,0.981404


Approx remaining CV TRAIN minutes (excludes validation): 62.4

EfficientNet fold 5/5
fold5 epoch 1 / 18 loss 0.73865 seconds 207.8
fold5 epoch 2 / 18 loss 0.39001 seconds 207.8
fold5 epoch 3 / 18 loss 0.28509 seconds 212.7
fold5 epoch 4 / 18 loss 0.23898 seconds 209.9
fold5 epoch 5 / 18 loss 0.20023 seconds 207.5
fold5 epoch 6 / 18 loss 0.17203 seconds 205.4
fold5 epoch 7 / 18 loss 0.15233 seconds 204.7
fold5 epoch 8 / 18 loss 0.138 seconds 204.6
fold5 epoch 9 / 18 loss 0.12059 seconds 204.9
fold5 epoch 10 / 18 loss 0.11771 seconds 204.9
fold5 epoch 11 / 18 loss 0.10082 seconds 204.6
fold5 epoch 12 / 18 loss 0.094 seconds 209.4
fold5 epoch 13 / 18 loss 0.09015 seconds 217.8
fold5 epoch 14 / 18 loss 0.08068 seconds 210.6
fold5 epoch 15 / 18 loss 0.07268 seconds 212.8
fold5 epoch 16 / 18 loss 0.06977 seconds 213.4
fold5 epoch 17 / 18 loss 0.06896 seconds 212.5
fold5 epoch 18 / 18 loss 0.06841 seconds 213.1
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1
stress fold

suite,compound,original,single
procedure,,,
baseline,0.919297,0.980762,0.973996
blend50,0.940842,0.984526,0.981418
efficientnet,0.936064,0.981459,0.977605


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.44


In [ ]:
oof={};scores=[];overlaps=[]
started=time.monotonic()
for fold in range(1,6):
    print(f'\nEfficientNet fold {fold}/5',flush=True)
    oof[fold]=training.run_fold(RAW,groups,folds,NAME,fold,OUT,BATCH_SIZE,NUM_WORKERS)
    score,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold])
    scores.append(score);overlaps.append(overlap)
    display(evaluation.summarize(score))
    history=json.loads((OUT/NAME/f'fold{fold}'/'history.json').read_text())
    seconds=np.mean([h['seconds'] for h in history[-3:]])
    print('Approx remaining CV TRAIN minutes (excludes validation):',round(seconds*18*(5-fold)/60,1))
all_scores=pd.concat(scores,ignore_index=True)
all_overlap=pd.concat(overlaps,ignore_index=True)
all_scores.to_parquet(OUT/'development_scores.parquet',index=False)
all_overlap.to_parquet(OUT/'development_overlap.parquet',index=False)
print('CV elapsed hours:',round((time.monotonic()-started)/3600,2))


EfficientNet fold 1/5
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /Users/meng/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 8.23MB/s]


fold1 epoch 1 / 18 loss 0.75933 seconds 214.4
fold1 epoch 2 / 18 loss 0.40118 seconds 206.2
fold1 epoch 3 / 18 loss 0.30187 seconds 205.6
fold1 epoch 4 / 18 loss 0.23698 seconds 205.8
fold1 epoch 5 / 18 loss 0.20993 seconds 206.1
fold1 epoch 6 / 18 loss 0.18377 seconds 204.7
fold1 epoch 7 / 18 loss 0.15754 seconds 204.9
fold1 epoch 8 / 18 loss 0.14063 seconds 204.9
fold1 epoch 9 / 18 loss 0.13002 seconds 205.1
fold1 epoch 10 / 18 loss 0.11226 seconds 205.2
fold1 epoch 11 / 18 loss 0.10062 seconds 205.2
fold1 epoch 12 / 18 loss 0.09453 seconds 206.1
fold1 epoch 13 / 18 loss 0.08585 seconds 206.1
fold1 epoch 14 / 18 loss 0.0831 seconds 207.0
fold1 epoch 15 / 18 loss 0.07908 seconds 205.6
fold1 epoch 16 / 18 loss 0.07341 seconds 207.5
fold1 epoch 17 / 18 loss 0.07428 seconds 208.8
fold1 epoch 18 / 18 loss 0.07123 seconds 208.0
stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fol

suite,compound,original,single
procedure,,,
baseline,0.929587,0.981367,0.975196
blend50,0.944725,0.986331,0.977201
efficientnet,0.937731,0.981883,0.973776


Approx remaining CV TRAIN minutes (excludes validation): 249.7

EfficientNet fold 2/5
fold2 epoch 1 / 18 loss 0.75993 seconds 209.0
fold2 epoch 2 / 18 loss 0.3767 seconds 206.4
fold2 epoch 3 / 18 loss 0.29719 seconds 203.8
fold2 epoch 4 / 18 loss 0.23959 seconds 205.8
fold2 epoch 5 / 18 loss 0.20431 seconds 204.9
fold2 epoch 6 / 18 loss 0.17661 seconds 205.1
fold2 epoch 7 / 18 loss 0.16118 seconds 206.6
fold2 epoch 8 / 18 loss 0.14294 seconds 205.2
fold2 epoch 9 / 18 loss 0.12619 seconds 205.4
fold2 epoch 10 / 18 loss 0.11078 seconds 205.2
fold2 epoch 11 / 18 loss 0.10463 seconds 205.1
fold2 epoch 12 / 18 loss 0.09125 seconds 205.0
fold2 epoch 13 / 18 loss 0.08875 seconds 205.1
fold2 epoch 14 / 18 loss 0.08198 seconds 205.0
fold2 epoch 15 / 18 loss 0.07566 seconds 204.8
fold2 epoch 16 / 18 loss 0.07024 seconds 205.1
fold2 epoch 17 / 18 loss 0.07185 seconds 205.1
fold2 epoch 18 / 18 loss 0.07405 seconds 205.4
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress 

suite,compound,original,single
procedure,,,
baseline,0.936276,0.980392,0.982209
blend50,0.955820,0.983375,0.985600
efficientnet,0.956336,0.980697,0.984579


... (3 outputs left out)

suite,compound,original,single
procedure,,,
baseline,0.925264,0.978785,0.972586
blend50,0.942175,0.980890,0.981004
efficientnet,0.942818,0.979768,0.981404


Approx remaining CV TRAIN minutes (excludes validation): 62.4

EfficientNet fold 5/5
fold5 epoch 1 / 18 loss 0.73865 seconds 207.8
fold5 epoch 2 / 18 loss 0.39001 seconds 207.8
fold5 epoch 3 / 18 loss 0.28509 seconds 212.7
fold5 epoch 4 / 18 loss 0.23898 seconds 209.9
fold5 epoch 5 / 18 loss 0.20023 seconds 207.5
fold5 epoch 6 / 18 loss 0.17203 seconds 205.4
fold5 epoch 7 / 18 loss 0.15233 seconds 204.7
fold5 epoch 8 / 18 loss 0.138 seconds 204.6
fold5 epoch 9 / 18 loss 0.12059 seconds 204.9
fold5 epoch 10 / 18 loss 0.11771 seconds 204.9
fold5 epoch 11 / 18 loss 0.10082 seconds 204.6
fold5 epoch 12 / 18 loss 0.094 seconds 209.4
fold5 epoch 13 / 18 loss 0.09015 seconds 217.8
fold5 epoch 14 / 18 loss 0.08068 seconds 210.6
fold5 epoch 15 / 18 loss 0.07268 seconds 212.8
fold5 epoch 16 / 18 loss 0.06977 seconds 213.4
fold5 epoch 17 / 18 loss 0.06896 seconds 212.5
fold5 epoch 18 / 18 loss 0.06841 seconds 213.1
stress fold5 gaussian_1
stress fold5 gaussian_2
stress fold5 speckle_1
stress fold

suite,compound,original,single
procedure,,,
baseline,0.919297,0.980762,0.973996
blend50,0.940842,0.984526,0.981418
efficientnet,0.936064,0.981459,0.977605


Approx remaining CV TRAIN minutes (excludes validation): 0.0
CV elapsed hours: 5.44


## 4. Evidence และเลือกสูตรจาก development เท่านั้น
เกณฑ์เป็น project heuristic: original ลดไม่เกิน .001; stress แบบเฉลี่ยเท่ากันต่อ family เพิ่ม ≥.002 และชนะ ≥4/5 folds; per-class ลดไม่เกิน .01; condition ลดไม่เกิน .02. ไม่บังคับ single-corruption เพิ่ม .005 เหมือน gate เก่า เพราะใกล้เพดาน แต่ยังจำกัด regression.

In [5]:
display(evaluation.summarize(all_scores))
display(all_overlap[all_overlap.condition.eq('original')])
selection=evaluation.select(all_scores)
validation.write_json(OUT/'development_selection.json',selection)
display(selection)
# Class-specific OOF and difficult quality slice evidence (no Test)
quality=pd.read_parquet(ROOT/'Task4/eda/advanced_train_v1/accounts.parquet').set_index('id')
rows=[]
for fold,frame in oof.items():
    base_frames=[pd.read_parquet(p.parent/'oof.parquet') for p in evaluation.references(ROOT,fold)]
    bp=np.mean([training.probabilities(f,'tta') for f in base_frames],axis=0)
    cp=training.probabilities(frame,'tta');q=quality.loc[frame.id]
    # Derive slice threshold from TRAINING fold quality only, then apply to held-out.
    fit,_=folds[fold-1]
    cutoff=quality.loc[groups.iloc[fit].id,'gradient_y'].quantile(.2)
    for proc,prob in [('baseline',bp),('efficientnet',cp),('blend50',(bp+cp)/2)]:
        masks={'all':np.ones(len(frame),dtype=bool),'low_gradient':q.gradient_y.to_numpy()<=cutoff}
        for label in range(5):masks[f'class_{label}']=frame.true.to_numpy()==label
        for label,mask in masks.items():
            rows.append({'fold':fold,'procedure':proc,'slice':label,'rows':int(mask.sum()),
                         'errors':int((prob.argmax(1)[mask]!=frame.true.to_numpy()[mask]).sum())})
slices=pd.DataFrame(rows);slices.to_parquet(OUT/'quality_class_slices.parquet',index=False)
summary=slices.groupby(['procedure','slice'])[['rows','errors']].sum()
summary['error_rate']=summary.errors/summary.rows
display(summary)

suite,compound,original,single
procedure,,,
baseline,0.926588,0.980847,0.975847
blend50,0.946368,0.983914,0.981367
efficientnet,0.942971,0.980419,0.977839


,fold,phase,condition,rows,baseline_errors,candidate_errors,both_wrong,candidate_fixes,candidate_breaks
0,1,development,original,6310,95,95,40,55,55
17,2,development,original,6309,105,98,52,53,46
34,3,development,original,6309,95,120,56,39,64
51,4,development,original,6309,108,105,52,56,53
68,5,development,original,6309,102,97,46,56,51


{'chosen': 'blend50',
 'reports': {'efficientnet': {'original_gain': -0.0004278798784256077,
   'stress_family_gain': 0.004390416891608354,
   'stress_fold_wins': 5,
   'worst_class_gain': -0.002989158014497817,
   'worst_condition_gain': -0.0012426350592488645,
   'passed': True},
  'blend50': {'original_gain': 0.0030669438521297556,
   'stress_family_gain': 0.007896884732937096,
   'stress_fold_wins': 5,
   'worst_class_gain': 0.0009649948486484706,
   'worst_condition_gain': 0.0030669438521297556,
   'passed': True}},
 'rule': 'original >= -0.001; family-balanced stress >= +0.002; wins >=4/5; per-class >= -0.01; condition >= -0.02',
 'exploratory_reused_oof': True}

rows  errors  error_rate
procedure    slice                                  
baseline     all           31546     505    0.016008
             class_0       12429     121    0.009735
             class_1        4407     100    0.022691
             class_2        5596      73    0.013045
             class_3        5926     116    0.019575
             class_4        3188      95    0.029799
             low_gradient   6308     216    0.034242
blend50      all           31546     421    0.013346
             class_0       12429     104    0.008368
             class_1        4407      71    0.016111
             class_2        5596      74    0.013224
             class_3        5926     100    0.016875
             class_4        3188      72    0.022585
             low_gradient   6308     199    0.031547
efficientnet all           31546     515    0.016325
             class_0       12429     140    0.011264
             class_1        4407      81    0.018380
             class_2        5596     102    0.018227
             class_3        5926     115    0.019406
             class_4        3188      77    0.024153
             low_gradient   6308     245    0.038840

## 5. Fixed audit veto
ประเมินสูตรที่เลือกไว้แล้ว ไม่เลือกผู้ชนะใหม่ตาม audit หากไม่ผ่านจะเก็บ baseline. ชุด audit ถูกใช้ในงานก่อนหน้าแล้ว ไม่ถือเป็น untouched holdout.

In [6]:
audit_result={'passed':False,'reason':'Development did not select a candidate'}
if selection['chosen']!='baseline':
    audit_parts=[]
    for fold in range(1,6):
        score,_=evaluation.compare(ROOT,RAW,groups,folds,OUT,fold,oof[fold],phase='audit')
        audit_parts.append(score)
    audit_scores=pd.concat(audit_parts,ignore_index=True)
    audit_scores.to_parquet(OUT/'audit_scores.parquet',index=False)
    audit_result=evaluation.audit(audit_scores,selection['chosen'])
    display(evaluation.summarize(audit_scores))
validation.write_json(OUT/'audit_decision.json',audit_result)
display(audit_result)

stress fold1 shot_1
stress fold1 shot_2
stress fold1 impulse_1
stress fold1 impulse_2
stress fold1 compound_audit_dark_jpeg
stress fold1 compound_audit_color_blur
stress fold2 shot_1
stress fold2 shot_2
stress fold2 impulse_1
stress fold2 impulse_2
stress fold2 compound_audit_dark_jpeg
stress fold2 compound_audit_color_blur
stress fold3 shot_1
stress fold3 shot_2
stress fold3 impulse_1
stress fold3 impulse_2
stress fold3 compound_audit_dark_jpeg
stress fold3 compound_audit_color_blur
stress fold4 shot_1
stress fold4 shot_2
stress fold4 impulse_1
stress fold4 impulse_2
stress fold4 compound_audit_dark_jpeg
stress fold4 compound_audit_color_blur
stress fold5 shot_1
stress fold5 shot_2
stress fold5 impulse_1
stress fold5 impulse_2
stress fold5 compound_audit_dark_jpeg
stress fold5 compound_audit_color_blur


suite,compound,single
procedure,,
baseline,0.927900,0.949621
blend50,0.946265,0.967599
efficientnet,0.941005,0.965088


{'passed': True,
 'mean_gain': 0.018106608655982796,
 'worst_condition_gain': 0.002796138343596599,
 'note': 'Fixed audit veto only; reused accounts/conditions, not independent holdout'}

## 6. Full refit และบันทึกโมเดล
ฝึก EfficientNet อีก 18 epochs บน Train ทั้งหมด เฉพาะเมื่อ development/audit ผ่าน. ไม่มี Test inference ในเล่มนี้; ส่งผลกลับให้ตรวจหลักฐานก่อนสร้าง submission. หากเลือก blend50 จะใช้ full compound/sampler เดิมร่วมกับ EfficientNet ใหม่ด้วยน้ำหนัก .25/.25/.50.

In [7]:
full_path=None
if RUN_FULL_REFIT and selection['chosen']!='baseline' and audit_result['passed']:
    full_path=training.fit_fixed(RAW,groups,training.RECIPES[NAME],OUT/NAME/'full',BATCH_SIZE,NUM_WORKERS)
    validation.write_json(OUT/'ready_for_review.json',{
        'procedure':selection['chosen'],'efficientnet_model':str(full_path),'sha256':validation.sha(full_path),
        'test_used':False,'submission_created':False,
        'selection':selection,'audit':audit_result})
    print('Saved full model:',full_path)
else:
    print('No full refit: inspect development/audit results above. Existing baseline preserved.')

full epoch 1 / 18 loss 0.69645 seconds 263.0
full epoch 2 / 18 loss 0.35972 seconds 260.2
full epoch 3 / 18 loss 0.27463 seconds 270.0
full epoch 4 / 18 loss 0.22365 seconds 266.8
full epoch 5 / 18 loss 0.18307 seconds 264.8
full epoch 6 / 18 loss 0.16303 seconds 268.0
full epoch 7 / 18 loss 0.14021 seconds 262.9
full epoch 8 / 18 loss 0.13073 seconds 261.2
full epoch 9 / 18 loss 0.11574 seconds 260.8
full epoch 10 / 18 loss 0.09927 seconds 259.6
full epoch 11 / 18 loss 0.09144 seconds 259.5
full epoch 12 / 18 loss 0.08179 seconds 264.7
full epoch 13 / 18 loss 0.07746 seconds 266.7
full epoch 14 / 18 loss 0.07026 seconds 266.2
full epoch 15 / 18 loss 0.06496 seconds 266.5
full epoch 16 / 18 loss 0.06303 seconds 269.3
full epoch 17 / 18 loss 0.06342 seconds 264.8
full epoch 18 / 18 loss 0.06516 seconds 261.5
Saved full model: /Users/meng/dev/MyUniversity/CPE342-MachineLearning/CPE342-Kaggle/Task4/models/results/backbone_diversity_v1/efficientnet_compound18/full/model.pt
